In [ ]:
import re
import spacy

# Charger modèle français
nlp = spacy.load("fr_core_news_sm")

text = """
Monsieur Dupont, 67 ans, est admis aux urgences le 14 mars 2023 pour dyspnée aiguë.
Antécédents : diabète de type 2, hypertension.
Le patient a été transféré en cardiologie le 16 mars 2023.
À l’examen biologique : CRP à 12.4 mg/L (norme < 5 mg/L).
Hémoglobine 11.2 g/dL (référence 13.0 - 17.0 g/dL).
Conclusion : suspicion d’insuffisance cardiaque.
"""

doc = nlp(text)

# =========================
# INITIALISATION SORTIE
# =========================
result = {
    "name": None,
    "age": None,
    "admission_date": None,
    "transfer_date": None,
    "antecedents": [],
    "biological_exams": [],
    "conclusion": None
}

# =========================
# 1️⃣ NOM + AGE
# =========================

for ent in doc.ents:
    if ent.label_ == "PER":
        result["name"] = ent.text
        break

age_match = re.search(r'(\d+)\s+ans', text)
if age_match:
    result["age"] = int(age_match.group(1))


# =========================
# 2️⃣ DATES (regex + contexte phrase)
# =========================

date_pattern = r'\b\d{1,2}\s+\w+\s+\d{4}\b'

for sent in doc.sents:
    sent_text = sent.text.lower()
    date_match = re.search(date_pattern, sent.text)

    if date_match:
        if "admis" in sent_text:
            result["admission_date"] = date_match.group(0)
        if "transfér" in sent_text:
            result["transfer_date"] = date_match.group(0)


# =========================
# 3️⃣ ANTECEDENTS
# =========================

for sent in doc.sents:
    if "antécédents" in sent.text.lower():
        parts = sent.text.split(":")
        if len(parts) > 1:
            antecedents = parts[1]
            result["antecedents"] = [
                a.strip() for a in antecedents.split(",")
            ]


# =========================
# 4️⃣ EXAMENS BIOLOGIQUES
# =========================

exam_pattern = re.compile(
    r'(?P<name>[A-Za-zéèêàâç]+)\s*(?:à\s*)?'
    r'(?P<value>\d+(?:\.\d+)?)\s*'
    r'(?P<unit>mg/L|g/dL)'
)

interval_pattern = re.compile(
    r'(?:<\s*(?P<upper>\d+(?:\.\d+)?)|'
    r'(?P<low>\d+(?:\.\d+)?)\s*-\s*(?P<high>\d+(?:\.\d+)?))'
)

for sent in doc.sents:
    exam_match = exam_pattern.search(sent.text)

    if exam_match:
        exam_data = {
            "name": exam_match.group("name"),
            "value": float(exam_match.group("value")),
            "unit": exam_match.group("unit"),
            "reference": None
        }

        interval_match = interval_pattern.search(sent.text)
        if interval_match:
            if interval_match.group("upper"):
                exam_data["reference"] = [0.0, float(interval_match.group("upper"))]
            elif interval_match.group("low") and interval_match.group("high"):
                exam_data["reference"] = [
                    float(interval_match.group("low")),
                    float(interval_match.group("high"))
                ]

        result["biological_exams"].append(exam_data)


# =========================
# 5️⃣ CONCLUSION
# =========================

for sent in doc.sents:
    if "conclusion" in sent.text.lower():
        parts = sent.text.split(":")
        if len(parts) > 1:
            result["conclusion"] = parts[1].strip()


# =========================
# RESULTAT FINAL
# =========================

result